In [0]:
%run ./24-sliding-window

In [0]:
class SensorSummaryTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.kafka_bz")
        spark.sql("drop table if exists spark_stream.default.sensor_summary")

        spark.sql(f"CREATE TABLE spark_stream.default.kafka_bz(key STRING, value STRING)")

        dbutils.fs.rm(f"{self.checkpoint_dir}/sensor_summary", True)
        print("Done")
   
    def assertSensorSummary(self, id, start, end, expected_max):
        print(f"\tStarting Sensor Summary validation...", end='')
        actual_result = spark.table("spark_stream.default.sensor_summary").collect()
        expected_result = (spark.read.format("csv")
                                .option("header", "true")
                                .load(f"{self.base_data_dir}/datasets/results/sliding_window_result.csv")
                                .collect()
                            )
        assert expected_result == actual_result, f"Test failed! actual max is {actual_result}"
        print("Done")

    def runTests(self):
        self.cleanTests()

        stream = SlidingAggregate()
       
        print("\nTesting all events...") 
        spark.sql("""INSERT INTO spark_stream.default.kafka_bz VALUES
                  ('SET41', '{"CreatedTime": "2019-02-05 09:54:00","Reading": 36.2}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 09:59:00","Reading": 36.5}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:04:00","Reading": 36.8}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:09:00","Reading": 36.2}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:14:00","Reading": 36.5}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:19:00","Reading": 36.3}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:24:00","Reading": 37.7}'),
                  ('SET41', '{"CreatedTime": "2019-02-05 10:29:00","Reading": 37.2}')
            """)
        sQuery = stream.process()
        sQuery.awaitTermination()
        print("Validation passed.\n")    

In [0]:
ts = SensorSummaryTestSuite()
ts.runTests()